# MLflow: Experiment Tracking & Model Comparison
## MLOps Class — Taxi Tip Prediction

This notebook teaches **MLflow** by training and comparing multiple models on the NYC Taxi dataset from previous class sessions.

### What you'll learn
| Concept | MLflow API |
|---------|-----------|
| Create & name an experiment | `mlflow.set_experiment()` |
| Start a run, log params & metrics | `mlflow.start_run()`, `log_param()`, `log_metric()` |
| Save model artifacts | `mlflow.sklearn.log_model()` |
| Log plots/files as artifacts | `mlflow.log_artifact()` |
| One-line automatic logging | `mlflow.autolog()` |
| Compare runs programmatically | `mlflow.search_runs()` |
| Register & load the best model | `mlflow.register_model()`, `mlflow.sklearn.load_model()` |

### Task: Predict NYC Taxi Tip Amount (Regression)
Features: `trip_distance`, `fare_amount`, `passenger_count`, `duration_min`, `hour_of_day`, `day_of_week`  
Label: `tip_amount`

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — safe for notebooks & scripts

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

print(f"MLflow version : {mlflow.__version__}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")

MLflow version : 3.13.0
Tracking URI   : sqlite:////home/alka/MLOPs-Class/mlflow/mlflow.db


---
## 1. MLflow Core Concepts

```
Experiment
└── Run 1  (params: n_estimators=100, max_depth=5)
│   ├── Metrics:  rmse=1.23, r2=0.87
│   ├── Params:   n_estimators=100, ...
│   ├── Artifacts: model.pkl, feature_importance.png
│   └── Tags:     model_type=RandomForest
└── Run 2  (params: learning_rate=0.1, n_estimators=200)
    ├── Metrics:  rmse=1.10, r2=0.91
    └── ...
```

**Tracking Server** stores all of this. By default it writes to `./mlruns/` (file-based).  
You can also point it at a SQLite DB (`sqlite:///mlflow.db`) or a remote server.

In [3]:
# Point MLflow at the local tracking server running at http://localhost:5000
mlflow.set_tracking_uri("http://localhost:5000")

# Create (or reuse) an experiment — all our runs will live here
EXPERIMENT_NAME = "taxi_tip_model_comparison_1"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {EXPERIMENT_NAME}")

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"Experiment ID: {exp.experiment_id}")

Tracking URI : http://localhost:5000
Experiment   : taxi_tip_model_comparison_1
Experiment ID: 3


---
## 2. Load & Prepare Taxi Data

We use the **preprocessed taxi dataset** from the ML Frameworks class (`taxi_tip_regression.parquet`).  
It already contains engineered features: `duration_min`, `hour_of_day`, `day_of_week`.  
All rows are credit-card trips (`payment_type == 1`) with sensible ranges applied.

In [4]:
PARQUET_PATH = "/home/alka/MLOPs-Class/ML_Framework/taxi_tip_regression.parquet"

df = pd.read_parquet(PARQUET_PATH)
print(f"Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns       : {list(df.columns)}")
df.describe().round(3)

Dataset shape : 119,432 rows × 7 columns
Columns       : ['trip_distance', 'fare_amount', 'passenger_count', 'duration_min', 'hour_of_day', 'day_of_week', 'tip_amount']


,trip_distance,fare_amount,passenger_count,duration_min,hour_of_day,day_of_week,tip_amount
count,119432.000,119432.000,119432.000,119432.000,119432.000,119432.000,119432.000
mean,0.001,0.000,-0.010,0.005,0.014,0.006,4.159
std,0.996,0.992,0.994,0.986,1.008,0.999,3.672
min,-0.779,-0.937,-1.517,-1.328,-2.460,-1.513,0.000
25%,-0.529,-0.556,-0.404,-0.661,-0.549,-1.011,2.150
50%,-0.367,-0.344,-0.404,-0.264,0.145,-0.006,3.150
75%,-0.013,0.080,-0.404,0.355,0.840,0.998,4.700
max,10.480,10.931,5.161,8.979,1.534,1.500,56.710


In [5]:
FEATURES = ["trip_distance", "fare_amount", "passenger_count",
            "duration_min", "hour_of_day", "day_of_week"]
LABEL = "tip_amount"

X = df[FEATURES].values
y = df[LABEL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# The parquet already has scaled features, but we rescale here from raw
# so each model run in MLflow is fully reproducible from raw inputs.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")
print(f"Label mean : {y_train.mean():.3f}  |  std : {y_train.std():.3f}")

# Helper: compute all metrics at once
def compute_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "r2": r2}

print("\nMetrics helper ready.")

Train : 95,545  |  Test : 23,887
Label mean : 4.163  |  std : 3.668

Metrics helper ready.


---
## 3. Manual MLflow Logging — The Building Blocks

`mlflow.start_run()` opens a **run context**.  
Everything logged inside that `with` block belongs to the same run.

```python
with mlflow.start_run(run_name="my_run"):
    mlflow.log_param("alpha", 0.1)           # single param
    mlflow.log_params({"a": 1, "b": 2})      # dict of params
    mlflow.log_metric("rmse", 1.23)          # single metric
    mlflow.log_metrics({"rmse": 1.23, "r2": 0.9})  # dict of metrics
    mlflow.log_artifact("plot.png")          # file artifact
    mlflow.sklearn.log_model(model, "model") # serialise + store model
    mlflow.set_tag("model_type", "Ridge")    # searchable label
```

### 3.1 Model 1 — Linear Regression (baseline)

The simplest model, manually logged step by step so you can see exactly what each call does.

In [6]:
mlflow.end_run()  # make sure no run is active before starting a new one
with mlflow.start_run(run_name="LinearRegression") as run:
    # --- 1. Train ---
    lr = LinearRegression()
    lr.fit(X_train_s, y_train)
    preds = lr.predict(X_test_s)

    # --- 2. Log parameters (hyperparams / config) ---
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("fit_intercept", lr.fit_intercept)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("n_train_samples", X_train.shape[0])

    # --- 3. Log metrics (performance numbers) ---
    metrics = compute_metrics(y_test, preds)
    mlflow.log_metrics(metrics)

    # --- 4. Tag the run (free-text labels, searchable later) ---
    mlflow.set_tag("model_family", "linear")
    mlflow.set_tag("scaled_input", "yes")

    # --- 5. Log the trained model as an artifact ---
    mlflow.sklearn.log_model(lr, artifact_path="model")

    lr_run_id = run.info.run_id

print(f"Run ID : {lr_run_id}")
print(f"Metrics: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  R²={metrics['r2']:.4f}")

2026/06/19 16:41:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/19 16:41:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearRegression at: http://localhost:5000/#/experiments/3/runs/5d18d0720e084293886d6de8ec0c6988
🧪 View experiment at: http://localhost:5000/#/experiments/3
Run ID : 5d18d0720e084293886d6de8ec0c6988
Metrics: RMSE=2.2399  MAE=1.2911  R²=0.6317


### 3.2 Model 2 — Ridge Regression (regularized linear)

Same idea as LinearRegression but with L2 regularization (`alpha`).  
We log **multiple alpha values** to show how params differ across runs.

In [ ]:
ridge_run_ids = {}

for alpha in [0.01, 1.0, 10.0]:
    with mlflow.start_run(run_name=f"Ridge_alpha={alpha}") as run:
        model = Ridge(alpha=alpha)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)

        mlflow.log_params({
            "model_type"    : "Ridge",
            "alpha"         : alpha,
            "fit_intercept" : model.fit_intercept,
            "n_features"    : X_train.shape[1],
        })
        metrics = compute_metrics(y_test, preds)
        mlflow.log_metrics(metrics)
        mlflow.set_tag("model_family", "linear")
        mlflow.sklearn.log_model(model, artifact_path="model")

        ridge_run_ids[alpha] = run.info.run_id
        print(f"Ridge alpha={alpha:>5}  →  RMSE={metrics['rmse']:.4f}  R²={metrics['r2']:.4f}")

### 3.3 Model 3 — Random Forest + Artifact Logging

Here we log a **feature importance plot** as an artifact.  
Artifacts can be any file: plots, CSVs, HTML reports, pickled objects, etc.

In [ ]:
with mlflow.start_run(run_name="RandomForest") as run:
    params = {
        "model_type"   : "RandomForestRegressor",
        "n_estimators" : 100,
        "max_depth"    : 6,
        "max_features" : "sqrt",
        "random_state" : 42,
    }
    rf = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        max_features=params["max_features"],
        random_state=params["random_state"],
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)          # RF handles unscaled data fine
    preds = rf.predict(X_test)

    mlflow.log_params(params)
    metrics = compute_metrics(y_test, preds)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("model_family", "tree_ensemble")

    # --- Log feature importance as an artifact ---
    fig, ax = plt.subplots(figsize=(7, 4))
    importances = rf.feature_importances_
    idx = np.argsort(importances)[::-1]
    ax.bar(range(len(FEATURES)), importances[idx])
    ax.set_xticks(range(len(FEATURES)))
    ax.set_xticklabels([FEATURES[i] for i in idx], rotation=35, ha="right")
    ax.set_title("Random Forest — Feature Importances (Taxi Tip)")
    ax.set_ylabel("Importance")
    fig.tight_layout()
    artifact_path = "/tmp/rf_feature_importance.png"
    fig.savefig(artifact_path, dpi=100)
    plt.close(fig)
    mlflow.log_artifact(artifact_path, artifact_path="plots")  # stored under plots/

    # --- Log the model ---
    mlflow.sklearn.log_model(rf, artifact_path="model")

    rf_run_id = run.info.run_id

print(f"Run ID : {rf_run_id}")
print(f"Metrics: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  R²={metrics['r2']:.4f}")

### 3.4 Model 4 — XGBoost

XGBoost from the ML Frameworks class, now tracked in MLflow.  
We also log **per-epoch training loss** using `mlflow.log_metric(step=epoch)` to track convergence.

In [ ]:
with mlflow.start_run(run_name="XGBoost") as run:
    params = {
        "model_type"    : "XGBRegressor",
        "n_estimators"  : 200,
        "max_depth"     : 5,
        "learning_rate" : 0.1,
        "subsample"     : 0.8,
        "colsample_bytree": 0.8,
        "random_state"  : 42,
    }
    xgb_model = xgb.XGBRegressor(
        n_estimators    = params["n_estimators"],
        max_depth       = params["max_depth"],
        learning_rate   = params["learning_rate"],
        subsample       = params["subsample"],
        colsample_bytree= params["colsample_bytree"],
        objective       = "reg:squarederror",
        random_state    = params["random_state"],
        verbosity       = 0,
    )

    # Train with eval set so we can capture per-round RMSE
    eval_set = [(X_test, y_test)]
    xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

    mlflow.log_params(params)

    # Log per-round validation RMSE as a metric series (step = boosting round)
    evals_result = xgb_model.evals_result()
    for step, val_rmse in enumerate(evals_result["validation_0"]["rmse"]):
        mlflow.log_metric("val_rmse_per_round", val_rmse, step=step)

    preds = xgb_model.predict(X_test)
    metrics = compute_metrics(y_test, preds)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("model_family", "tree_ensemble")

    # Feature importance plot
    fig, ax = plt.subplots(figsize=(7, 4))
    xgb_imp = xgb_model.feature_importances_
    idx = np.argsort(xgb_imp)[::-1]
    ax.bar(range(len(FEATURES)), xgb_imp[idx])
    ax.set_xticks(range(len(FEATURES)))
    ax.set_xticklabels([FEATURES[i] for i in idx], rotation=35, ha="right")
    ax.set_title("XGBoost — Feature Importances (Taxi Tip)")
    ax.set_ylabel("Importance")
    fig.tight_layout()
    imp_path = "/tmp/xgb_feature_importance.png"
    fig.savefig(imp_path, dpi=100)
    plt.close(fig)
    mlflow.log_artifact(imp_path, artifact_path="plots")

    mlflow.sklearn.log_model(xgb_model, artifact_path="model")
    xgb_run_id = run.info.run_id

print(f"Run ID : {xgb_run_id}")
print(f"Metrics: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  R²={metrics['r2']:.4f}")

### 3.5 Model 5 — Gradient Boosting (sklearn)

sklearn's GBR is slower than XGBoost but comes built-in.  
Good to compare: same algorithm family, different implementation.

In [ ]:
with mlflow.start_run(run_name="GradientBoosting") as run:
    params = {
        "model_type"    : "GradientBoostingRegressor",
        "n_estimators"  : 150,
        "max_depth"     : 4,
        "learning_rate" : 0.05,
        "subsample"     : 0.8,
        "random_state"  : 42,
    }
    gbr = GradientBoostingRegressor(
        n_estimators = params["n_estimators"],
        max_depth    = params["max_depth"],
        learning_rate= params["learning_rate"],
        subsample    = params["subsample"],
        random_state = params["random_state"],
    )
    gbr.fit(X_train, y_train)
    preds = gbr.predict(X_test)

    mlflow.log_params(params)
    metrics = compute_metrics(y_test, preds)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("model_family", "tree_ensemble")

    # Log staged predictions to capture training curve
    for step, staged_pred in enumerate(gbr.staged_predict(X_test)):
        stage_rmse = mean_squared_error(y_test, staged_pred) ** 0.5
        mlflow.log_metric("staged_rmse", stage_rmse, step=step)

    mlflow.sklearn.log_model(gbr, artifact_path="model")
    gbr_run_id = run.info.run_id

print(f"Run ID : {gbr_run_id}")
print(f"Metrics: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  R²={metrics['r2']:.4f}")

---
## 4. MLflow Autolog — One Line to Track Everything

`mlflow.autolog()` instruments the sklearn/xgboost library automatically:  
- Logs all hyperparameters  
- Logs training metrics  
- Saves the model artifact  
- Generates an `estimator.html` diagram  

**No manual `log_param` / `log_metric` calls needed.**

In [ ]:
mlflow.autolog()   # ← single call, instruments sklearn automatically

with mlflow.start_run(run_name="RF_autolog") as run:
    rf_auto = RandomForestRegressor(
        n_estimators=200, max_depth=8, max_features="sqrt",
        random_state=42, n_jobs=-1
    )
    rf_auto.fit(X_train, y_train)          # autolog hooks into .fit()

    # Autolog already captured train metrics; add our own test metrics too
    preds = rf_auto.predict(X_test)
    metrics = compute_metrics(y_test, preds)
    mlflow.log_metrics({f"test_{k}": v for k, v in metrics.items()})
    mlflow.set_tag("logged_with", "autolog")

    auto_run_id = run.info.run_id

mlflow.autolog(disable=True)   # turn off so later cells use manual logging

print(f"Run ID  : {auto_run_id}")
print(f"Test    : RMSE={metrics['rmse']:.4f}  R²={metrics['r2']:.4f}")
print("\nAutolog logged all RF hyperparams automatically — check the UI!")

---
## 5. Comparing Runs Programmatically

`mlflow.search_runs()` returns a **pandas DataFrame** of all runs in an experiment.  
You can filter, sort, and plot directly in Python — no UI needed.

In [ ]:
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"],      # sort best RMSE first
)

# Select columns that matter for teaching
display_cols = [
    "tags.mlflow.runName",
    "params.model_type",
    "params.n_estimators",
    "params.max_depth",
    "params.alpha",
    "params.learning_rate",
    "metrics.rmse",
    "metrics.mae",
    "metrics.r2",
    "run_id",
]
available = [c for c in display_cols if c in runs_df.columns]
comparison = (
    runs_df[available]
    .rename(columns=lambda c: c.replace("tags.mlflow.", "").replace("params.", "").replace("metrics.", ""))
    .dropna(subset=["rmse"])
    .reset_index(drop=True)
)
comparison["rmse"]  = comparison["rmse"].round(4)
comparison["mae"]   = comparison["mae"].round(4)
comparison["r2"]    = comparison["r2"].round(4)
print(f"Total runs in experiment: {len(runs_df)}")
comparison

In [ ]:
# Search with a filter: only tree ensembles
tree_runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.model_family = 'tree_ensemble'",
    order_by=["metrics.rmse ASC"],
)
print(f"Tree ensemble runs: {len(tree_runs)}")
if "tags.mlflow.runName" in tree_runs.columns and "metrics.rmse" in tree_runs.columns:
    print(tree_runs[["tags.mlflow.runName", "metrics.rmse", "metrics.r2"]].to_string(index=False))

In [ ]:
# Visual comparison: RMSE and R² bar charts
plot_df = (
    mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["metrics.rmse ASC"],
    )
    .dropna(subset=["metrics.rmse"])
    [["tags.mlflow.runName", "metrics.rmse", "metrics.r2"]]
    .rename(columns={"tags.mlflow.runName": "run", "metrics.rmse": "RMSE", "metrics.r2": "R2"})
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(plot_df)))

# RMSE — lower is better
axes[0].barh(plot_df["run"], plot_df["RMSE"], color=colors)
axes[0].set_xlabel("RMSE  (lower is better)")
axes[0].set_title("Model Comparison — RMSE")
axes[0].invert_yaxis()
for i, v in enumerate(plot_df["RMSE"]):
    axes[0].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

# R² — higher is better
colors_r2 = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(plot_df)))
axes[1].barh(plot_df["run"], plot_df["R2"], color=colors_r2)
axes[1].set_xlabel("R²  (higher is better)")
axes[1].set_title("Model Comparison — R²")
axes[1].invert_yaxis()
for i, v in enumerate(plot_df["R2"]):
    axes[1].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

fig.suptitle("MLflow Experiment: Taxi Tip Prediction", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()
print("Best model (lowest RMSE):", plot_df.iloc[0]["run"])

---
## 6. Load a Saved Model from MLflow

Every logged model can be reloaded by **run ID** — no need to keep the Python object in memory.  
This is the core of MLflow's **reproducibility** guarantee.

In [ ]:
# Find the best run (lowest RMSE) from the experiment
best_run = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.rmse ASC"],
).iloc[0]

best_run_id   = best_run["run_id"]
best_run_name = best_run.get("tags.mlflow.runName", "unknown")
best_rmse     = best_run["metrics.rmse"]

print(f"Best run  : {best_run_name}")
print(f"Run ID    : {best_run_id}")
print(f"Best RMSE : {best_rmse:.4f}")

# Load the model from its run URI — mlflow handles deserialisation
model_uri = f"runs:/{best_run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)
print(f"\nLoaded model type: {type(loaded_model).__name__}")

# Verify it produces the same predictions
loaded_preds = loaded_model.predict(X_test)
loaded_rmse = mean_squared_error(y_test, loaded_preds) ** 0.5
print(f"Loaded model RMSE: {loaded_rmse:.4f}  (should match {best_rmse:.4f})")

---
## 7. Model Registry

The **Model Registry** is a central catalog of named, versioned models.  
It lets you promote a model through stages: `None → Staging → Production`.

```
Registry
└── "taxi_tip_predictor"
    ├── Version 1  (run_id=abc...) — stage: Staging
    └── Version 2  (run_id=xyz...) — stage: Production   ← the live model
```

This decouples **training** (data scientists) from **deployment** (MLOps/engineering).

In [ ]:
MODEL_NAME = "taxi_tip_predictor"

# Register the best run's model into the Registry
model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"Registered: '{MODEL_NAME}'")
print(f"Version   : {registered.version}")
print(f"Status    : {registered.status}")  # READY once registration completes

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Transition latest version to "Staging"
latest_version = registered.version
client.transition_model_version_stage(
    name    = MODEL_NAME,
    version = latest_version,
    stage   = "Staging",
)
print(f"Model '{MODEL_NAME}' v{latest_version} → Staging")

# Load directly from the registry using the stage alias
staging_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Staging")
staging_preds = staging_model.predict(X_test[:5])
print(f"\nSample predictions from Staging model: {staging_preds.round(2)}")
print(f"Actual tip amounts:                    {y_test[:5].round(2)}")

In [ ]:
# List all registered model versions
print(f"All versions of '{MODEL_NAME}':")
for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(f"  v{mv.version}  stage={mv.current_stage:10s}  run_id={mv.run_id[:8]}...")

---
## 8. MLflow UI

Launch the MLflow tracking UI to explore all runs visually:

```bash
# From the mlflow/ directory:
cd /home/alka/MLOPs-Class/mlflow
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open **http://localhost:5000** in your browser.

### What you can do in the UI
| Tab | What you see |
|-----|-------------|
| **Experiments** | all experiments, click to see runs |
| **Runs table** | sort/filter by any param or metric |
| **Run detail** | params, metrics, artifacts, model schema |
| **Artifacts** | download the `estimator.html`, feature importance plots |
| **Compare** | overlay metric charts across selected runs |
| **Models** | Registry: versions, stages, lineage |

### Key concepts recap

```
mlflow.set_experiment("name")          → create / switch experiment
mlflow.start_run(run_name="...")       → open a run context
  mlflow.log_param("k", v)            → one hyperparameter
  mlflow.log_metric("rmse", 1.23)     → one scalar metric
  mlflow.log_metric("loss", v, step=i)→ metric series (convergence curve)
  mlflow.log_artifact("file.png")     → attach any file
  mlflow.sklearn.log_model(m, "model")→ serialise + store model
  mlflow.set_tag("env", "dev")        → searchable label
mlflow.autolog()                       → auto-instrument library
mlflow.search_runs(experiment_ids=[]) → DataFrame of all runs
mlflow.sklearn.load_model("runs:/id/model")     → reload model
mlflow.register_model(uri, name)      → add to Registry
client.transition_model_version_stage(...)      → promote to Staging/Production
mlflow.sklearn.load_model("models:/name/stage") → load from Registry
```

In [ ]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("taxi_tip_model_comparison")
client.delete_experiment(exp.experiment_id)  # soft-delete if not already

# Permanently purge from DB
# Run this in terminal:
# mlflow gc --backend-store-uri sqlite:///mlflow.db
